In [ ]:
/*
========================================================
GENERATING UNIQUE IDS - SIMPLE HINGLISH NOTES
========================================================

INTRO
--------------------------------------------------------
Har entity (user, order, message, shortened URL) ko ek 
identifier chahiye. Single database pe, ek AUTO-INCREMENTING 
column ye acche se handle karta. Database ek counter rakhta, 
har insert pe next value de deta.

PROBLEM TAB AATA JAB MULTIPLE INDEPENDENT WRITERS HOTE HAIN. 
Agar har writer apna LOCAL counter rakhe aur ID space 
partition na ho, to 2 writers SAME value produce kar sakte. 
IDs unique nahi rahenge.

INTERVIEW SCALE PE, ID generation sirf uniqueness ki baat 
nahi hai. Ordering, length, predictability, database index 
locality, clock behavior, aur operational failure modes bhi 
socho.

TEXT DIAGRAM - COLLISION PROBLEM:

  Writer A -> ID: 1001 --\
                           ---> Storage -> "Duplicate ID!" 
  Writer B -> ID: 1001 --/       (COLLISION)

ACHA INTERVIEW ANSWER: Ek "universal best scheme" naam mat 
lo, TRADE-OFFS compare karo.


INTERVIEW ANSWER SHAPE
--------------------------------------------------------
Requirements se ID scheme choose karo:

1. Clarify karo: uniqueness scope, generation rate, ID 
   length, sortability, aur IDs public hain kya
2. AUTO-INCREMENT se start karo jab ek primary writer ho 
   aur predictable public IDs theek ho
3. UUID v4, UUID v7, ya ULID use karo jab decentralized 
   generation compact 64-bit IDs se zyada matter kare
4. SNOWFLAKE-STYLE IDs use karo jab compact, time-sortable 
   IDs chahiye aur machine IDs safely assign kar sakte ho
5. Operational safeguards mention karo: unique constraints, 
   random/truncated IDs ke liye collision handling, node ID 
   assignment, clock-backward behavior


YE PROBLEM HARD KYU HAI?
--------------------------------------------------------
Single primary database pe, unique ID generation easy hai. 
Database ek counter maintain karta, har insert pe increment 
karta, atomically AUTO_INCREMENT/SERIAL/sequence se handle 
karta. Har ID us table/sequence ke andar unique hoti.

TEXT DIAGRAM:

  [Single Server]
    Counter: 1 -> 2 -> 3 -> 4 -> ...

Jaise hi DOOSRA independent counter add karte ho, ye 
approach extra design maangta. Bina coordination ya 
partitioned ID space ke, DONO writers apne counters same 
value tak increment kar sakte.

TEXT DIAGRAM:

  Server A: Counter 1 -> 2 -> 3 -> "ID: 2"
  Server B: Counter 1 -> 2 -> 3 -> "ID: 2"
       |
  "SAME ID!" (Collision)

--------------------------------------------------------
THE COORDINATION DILEMMA
--------------------------------------------------------
Ek solution: COORDINATION add karo. ID generate karne se 
pehle, har writer ek central authority se check kare. Ye 
kaam kar sakta, but coordination ki cost hoti:

- LATENCY: Har ID generation ab network round-trip maangta. 
  Millions IDs/sec pe, ye overhead badi ban jaata
- SINGLE POINT OF FAILURE: Central authority down ho to, 
  koi bhi server naye IDs generate nahi kar sakta, sab 
  writes ruk jaate
- THROUGHPUT BOTTLENECK: Central authority limit karta poora 
  system kitna fast chal sakta hai

TEXT DIAGRAM:

  Server1 --\
  Server2 --- [Central Authority] -- Latency, Single Point 
  Server3 --/                        of Failure, Bottleneck

NOTE: Bahut saare high-scale ID schemes HOT PATH pe 
coordination avoid karte. Har generator independently IDs 
create kar sakta kyunki design ID space partition karta ya 
collisions negligibly unlikely banata.

--------------------------------------------------------
DIFFERENT SYSTEMS KO ALAG NEEDS HOTI HAIN
--------------------------------------------------------
  Requirement       Description                Example Use Case
  -----------------------------------------------------------------
  Uniqueness        Sab servers/time mein no    Har system ko 
                     duplicates                  chahiye
  Sortable          Creation time se order       Time-series data, 
                     kar sakein                   activity feeds
  Compact           Chhota storage footprint     URL shorteners, 
                                                   mobile apps
  Unpredictable     Next/previous ID guess na    User-facing IDs, 
                     ho paye                       security
  High throughput   Millions/second generate     Social media, 
                                                   high-scale
  No coordination   Independently generate       Microservices, 
                     kar sakein                    distributed DBs

Koi single approach sab requirements satisfy nahi karta. 
Practical skill hai: apne system ke liye important 
trade-offs choose karna.


APPROACH 1: DATABASE AUTO-INCREMENT
--------------------------------------------------------
Sabse simple approach: database ko ID generation handle 
karne do.

```sql
CREATE TABLE users (
    id BIGINT AUTO_INCREMENT PRIMARY KEY,
    ...
);
```

Database internally counter maintain karta. Har insert 
atomically increment karta, aur counter restart ke baad bhi 
persist rehta. Single primary database ke liye, ye usually 
SIMPLEST CORRECT choice hai.

--------------------------------------------------------
MULTIPLE DATABASES KE SAATH SCALE KARNA
--------------------------------------------------------

STEP-BASED APPROACH:

TEXT DIAGRAM:

  Server 1: Odd IDs  -> 1, 3, 5, 7
  Server 2: Even IDs -> 2, 4, 6, 8

Server 1 odd numbers generate karta (start=1, step=2), 
Server 2 even (start=2, step=2). Ye overlap nahi karte jab 
tak offsets/step size sahi configured hain aur koi 2 
writers same offset share nahi karte.

RANGE-BASED APPROACH:
Har server ko ek DEDICATED RANGE mil jata, jisme wo 
independently operate karta.

TRADE-OFFS:

  ACHA (PROS)                        BURA (CONS)
  -----------------------------------------------------
  Implement/samajhna simple          Har ID ke liye database 
                                       round-trip chahiye
  Ek writer pe sequential + naturally Database single point of 
  sortable                             failure ban sakta (agar 
                                        ek primary pe depend)
  Compact (64-bit integer)           Horizontally scale karna 
                                       hard
  Human-readable                     Predictable (public IDs 
                                       ke liye security concern)

KAB YE APPROACH THEEK HAI:
Auto-increment small-medium scale systems ke liye achha 
kaam karta jaha already ek primary database hai aur 
horizontal scaling ki zaroorat nahi. Internal systems ke 
liye bhi theek hai jaha predictability concern nahi.

BUT agar microservices with independent databases bana rahe 
ho, ya high scale pe database dependencies ke bina IDs 
generate karne hon, to kuch aur chahiye.


APPROACH 2: UUID (UNIVERSALLY UNIQUE IDENTIFIER)
--------------------------------------------------------
UUIDs 128-bit identifiers hain jo bina coordination ke 
unique hone ke liye design kiye gaye.

USEFUL PROPERTY: Koi bhi server independently generate kar 
sakta. Random UUIDs ke liye, collision probability itni kam 
hai ki most systems negligible maante, phir bhi final 
guardrail ke tor pe database uniqueness constraint rakhte.

--------------------------------------------------------
UUID VERSIONS
--------------------------------------------------------
  Version   Kaise Banta          Property
  ----------------------------------------------------------
  UUID v1   Timestamp + MAC      Sortable by time (BUT split 
                                   across fields hai, isliye 
                                   raw sort se time order nahi 
                                   milta), MAC address expose 
                                   karta
  UUID v4   Random (122 bits)    Most common, NOT sortable
  UUID v7   Timestamp + Random   Time-ordered, achhi database 
                                   locality

UUID v1: Current timestamp + machine ka MAC address combine 
karta. Timestamp kai fields (time_low, time_mid, time_hi) 
mein split hai, isliye v1 values raw strings/bytes ki tarah 
chronologically sort NAHI hote; time order nikalne ke liye 
fields rearrange karne padte. Hardware MAC address embed 
karta - privacy concern ho sakta.

UUID v4: Sabse zyada use hone wala version. 122 random bits 
generate karta. Simple, widely supported, but TIME-SORTABLE 
NAHI hai.

UUID v7: Unix timestamp (milliseconds) + random bits combine 
karta. Time-ordering milta bina hardware info expose kiye. 
Naye systems ke liye strong choice jinhe decentralized 
generation + UUID v4 se better index locality chahiye.

--------------------------------------------------------
COLLISION PROBABILITY
--------------------------------------------------------
122 random bits ke saath, UUID v4 mein collision probability 
NEGLIGIBLE hai. Practically, UUID v4 collisions itne unlikely 
hain ki usually limiting risk nahi banti. Phir bhi, ek unique 
constraint sasta hai aur implementation bugs, bad random 
number generators, ya accidental reuse catch karta.

--------------------------------------------------------
DATABASE PERFORMANCE PROBLEM
--------------------------------------------------------
Random UUIDs ka ek database drawback hai: INDEX LOCALITY 
KHARAB kar sakte hain.

Jab sequential IDs primary keys hote, naye inserts hamesha 
B-tree index ke END mein jaate. Database efficiently 
append kar sakta bina existing pages reorganize kiye.

Random UUIDs, dusri taraf, inserts ko index mein HAR JAGAH 
bhejte hain.

TEXT DIAGRAM:

  RANDOM UUIDs (Inefficient):
    Insert 550e... -> Middle of index
    Insert a29b... -> Beginning of index
    Insert 1d4a... -> Different location

  SEQUENTIAL IDs (Efficient):
    Insert 1 -> End of index
    Insert 2 -> End of index
    Insert 3 -> End of index

Ye random access pattern PAGE SPLITS, FRAGMENTATION, aur 
zyada I/O cause kar sakta. Exact impact database engine, 
storage layout, fill factor, aur write rate pe depend karta, 
but UUID v4 primary keys usually sequential/time-ordered 
IDs se WORSE hote sustained insert locality ke liye.

TRADE-OFFS:

  ACHA (PROS)                        BURA (CONS)
  -----------------------------------------------------
  Coordination ki zaroorat nahi      Bada size (128 bits = 
                                       16 bytes)
  Achhi randomness ke saath          Sortable nahi (v4)
  negligible collision risk
  Universally supported              Database index 
                                       performance kharab
  Kahi bhi generate ho sakte         Human-readable nahi

KAB YE APPROACH THEEK HAI:
UUIDs distributed systems ke liye achhe hain jaha IDs bina 
kisi coordination ke generate hone chahiye, especially jab 
ID generation CLIENT-SIDE pe ho. Multiple sources se data 
merge karte waqt bhi useful hain.

WARNING: URL shorteners ke liye plain UUID strings avoid 
karo jaha visible code SHORT hona chahiye. High-write 
databases ke liye, UUID v4 ke bajaye UUID v7, ULID, ya kisi 
time-ordered format ko prefer karo jab index locality 
matter kare.


APPROACH 3: SNOWFLAKE IDs
--------------------------------------------------------
Twitter ne 2010s mein ye saare challenges face kiye. Unhe 
aise IDs chahiye the jo:
- HAZAARON servers mein unique hon
- Time se sortable hon (tweets chronologically dikhane ke 
  liye)
- 64 bits mein fit ho jaayein
- Bina central coordination ke generate ho sakein

Unka solution "SNOWFLAKE" bahut widely copy hua. Discord, 
Instagram, aur bahut saari high-scale services similar 
timestamp-node-sequence formats use karte.

--------------------------------------------------------
STRUCTURE SAMJHO
--------------------------------------------------------
Ek Snowflake ID 64 bits mein 4 fields encode karta:

TEXT DIAGRAM:

  [Sign: 1 bit (hamesha 0)]
  [Timestamp: 41 bits (ms since epoch)]
  [Machine ID: 10 bits (1,024 machines)]
  [Sequence: 12 bits (4,096 per ms)]

- SIGN BIT (1 bit): Hamesha zero. Ensure karta ID hamesha 
  POSITIVE number ho.
- TIMESTAMP (41 bits): Ek custom epoch se milliseconds. 
  Twitter Nov 4, 2010 use karta apna epoch. 41 bits ~69 
  saal ka IDs de deta.
- MACHINE ID (10 bits): Kaunsa server ID generate kar raha, 
  identify karta. 1,024 unique machines ek saath IDs 
  generate kar sakte.
- SEQUENCE (12 bits): Same millisecond, same machine pe 
  generate hue IDs ka counter. 4,096 IDs per millisecond 
  per machine support karta.

--------------------------------------------------------
PRACTICE MEIN KAISE KAAM KARTA
--------------------------------------------------------
Har node pe generation logic straightforward hai (pseudocode 
samjhein): current timestamp lo, agar same ms mein pehle se 
IDs bane hain to sequence increment karo, warna sequence 
reset karo, phir sab fields ko bit-shift karke combine karo 
(sign + timestamp + machine_id + sequence).

Har machine independently operate karta. Jab tak machine IDs 
UNIQUE hain, clock BACKWARD nahi jaata, aur per-millisecond 
sequence OVERFLOW nahi hota bina wait kiye, IDs collide 
nahi karenge. Timestamp component IDs ko roughly creation 
time se sortable banata.

--------------------------------------------------------
THROUGHPUT CAPACITY
--------------------------------------------------------
TEXT DIAGRAM:

  Per Machine: 4,096 IDs/ms = 4M IDs/sec
  Per Cluster: 1,024 machines = 4B IDs/sec
  Time Range: 41 bits = 69 years

Ek machine 4,096 IDs/millisecond generate kar sakta agar 
wo rate sustain kar sake aur sequence overflow pe wait kare. 
1,024 machines ka cluster theoretically ~4 billion IDs/sec 
tak generate kar sakta, though real systems CPU, networking, 
storage, aur downstream writes se limited hote.

--------------------------------------------------------
IDs SE INFORMATION EXTRACT KARNA
--------------------------------------------------------
Snowflake IDs ka ek useful property: TIMESTAMP EXTRACT kar 
sakte ho (bit-shift karke). Ye debugging, auditing, aur 
entities kab bane ye estimate karne ke liye useful hai bina 
extra metadata query kiye.

--------------------------------------------------------
64-BIT SERIALIZATION TRAP (IMPORTANT!)
--------------------------------------------------------
Snowflake IDs FULL 64-BIT INTEGERS hain, aur ye API boundary 
pe ek PROBLEM create karta. JavaScript numbers ko IEEE 754 
doubles ki tarah represent karta, jo sirf 2^53 - 1 (~9 
quadrillion) tak EXACTLY hold kar sakta. Snowflake ID isse 
bada hota hai, isliye JSON number ki tarah parse karne wala 
browser/Node.js client SILENTLY low-order digits kho deta, 
aur read-back value server-stored value se match nahi 
karta.

Twitter ne apne public API mein YE EXACT ISSUE face kiya 
aur IDs ko STRING (id_str field) ki tarah bhi expose karna 
pada, numeric form ke saath. FIX: 64-bit IDs ko JSON mein 
STRINGS ki tarah serialize karo, raw numbers ki tarah nahi.

--------------------------------------------------------
CLOCK PROBLEM
--------------------------------------------------------
Snowflake mostly MONOTONIC system clocks pe depend karta. 2 
scenarios problem cause karte:

  CLOCK MOVES BACKWARD:
    - Server ka clock backward adjust ho jaye, generator 
      pehle use kiye timestamps produce kar sakta - 
      DUPLICATE IDs possible
    - Solution: Clock catch up hone tak generate karna 
      REFUSE karo, logical clock use karo, ya alert karke 
      node ko service se hata do

  CLOCK SKEW BETWEEN SERVERS:
    - Different servers ka time alag ho sakta, jisse 
      different servers ke IDs REAL creation time se 
      perfectly ordered nahi honge
    - Bahut saare feeds/event logs mein approximate 
      ordering acceptable hai; strict ordering ko alag 
      mechanism chahiye

TRADE-OFFS:

  ACHA (PROS)                        BURA (CONS)
  -----------------------------------------------------
  Compact (64-bit)                   Machine ID coordination 
                                       chahiye
  Time-sortable                      Clock synchronization pe 
                                       depend
  High throughput                    69-year limit epoch se
  ID generation ke liye central      Simpler approaches se 
  coordination nahi                   zyada complex

KAB YE APPROACH THEEK HAI:
Snowflake un high-scale services ke liye achha hai jinhe 
time-sortable, 64-bit IDs chahiye aur jo reliably unique 
machine IDs assign kar sakte.

Simple applications ke liye ZYADA hai, aur serverless/
ephemeral environments mein use karna hard hai jaha stable 
machine IDs assign karna mushkil hai.


APPROACH 4: ULID (UNIVERSALLY UNIQUE 
LEXICOGRAPHICALLY SORTABLE IDENTIFIER)
--------------------------------------------------------
ULID UUID (no coordination) ke best properties ko Snowflake 
ki sortability ke saath COMBINE karne ke liye banaya gaya.

--------------------------------------------------------
STRUCTURE
--------------------------------------------------------
ULID 128 bits ko 2 parts mein pack karta: 48-bit millisecond 
timestamp + 80 bits randomness, 26-character Base32 string 
mein encode hota.

TEXT DIAGRAM:

  01ARZ3NDEKTSV4RRFFQ69G5FAV (26 characters)
  
  [Timestamp: 48 bits (ms since Unix epoch)]
  [Randomness: 80 bits]

- TIMESTAMP (48 bits): Unix epoch se milliseconds. ~8,000 
  saal ka IDs deta - Snowflake ke 69 saal se BAHUT ZYADA.
- RANDOMNESS (80 bits): Random bits jo SAME millisecond ke 
  andar collisions negligibly unlikely banate (sahi 
  generate hone pe).
- ENCODING: Crockford's Base32 use karta, jo CASE-INSENSITIVE 
  hai aur ambiguous letters (I, L, O, U) avoid karta.

--------------------------------------------------------
KEY PROPERTIES
--------------------------------------------------------
Timestamp-FIRST structure ka matlab: STRING SORTING time 
ordering produce karti. ULIDs ko strings ki tarah sort kar 
sakte ho aur different milliseconds ke across chronological 
order mil jata. SAME millisecond ke andar, normal ULIDs 
random suffix se sort hote, exact creation order se nahi. 
Kuch libraries "MONOTONIC ULIDs" support karte, jo same 
millisecond mein random part increment karte local ordering 
preserve karne ke liye.

--------------------------------------------------------
ULID vs UUID COMPARISON
--------------------------------------------------------
  Property             UUID v4       ULID
  ----------------------------------------------
  Size                 128 bits      128 bits
  Sortable             No            Yes
  Timestamp extractable No           Yes
  String length        36 chars      26 chars
  Case-sensitive        No            No
  Coordination required No            No

ULID UUID-JAISI decentralized generation deta with SHORTER, 
TIME-SORTABLE string. Trade-off: timestamp VISIBLE hai, 
isliye jaha creation time HIDE karna chahiye wahan ideal 
nahi.

KAB YE APPROACH THEEK HAI:
ULID useful hai jab UUID-jaisi properties ke saath 
time-ordering chahiye. Distributed systems mein bina machine 
ID coordination ke achha kaam karta, aur lexicographic 
sorting property logs, events, database keys ke liye 
convenient hai.


APPROACH 5: MONGODB OBJECTID
--------------------------------------------------------
MongoDB ka default ID format cover karna zaroori hai kyunki 
ye widely use hota, aur UUID aur Snowflake ke design ke 
BEECH mein sits karta.

--------------------------------------------------------
STRUCTURE
--------------------------------------------------------
ObjectID 12 bytes (96 bits) hai, 3 fields mein divided. 
Layout timestamp ko PEHLE rakhta, taaki IDs roughly creation 
time se sort hon.

TEXT DIAGRAM:

  [Timestamp: 4 bytes (Unix seconds)]
  [Random: 5 bytes (per process)]
  [Counter: 3 bytes (incrementing)]

- TIMESTAMP (4 bytes): Unix timestamp (seconds). Second 
  granularity pe rough time-ordering deta.
- RANDOM VALUE (5 bytes): Har process ke liye ek baar 
  generate hota. Machine/process combination ke liye 
  unique.
- COUNTER (3 bytes): Incrementing counter, random value se 
  initialize hota. Counter wrap hone se pehle ~16 million 
  IDs/second per process support karta.

--------------------------------------------------------
KEY PROPERTIES
--------------------------------------------------------
ObjectID UUID se CHHOTA hai (12 bytes vs 16), roughly 
time-sortable, aur koi central coordinator zaroori nahi. 
Counter component same second ke andar bursts of IDs 
handle karta.

Ye format MongoDB ke document-oriented storage model ke 
liye optimized hai. Agar MongoDB use kar rahe ho, ObjectID 
usually DEFAULT hi hota jab tak koi specific requirement na 
ho.


APPROACH 6: TICKET SERVERS
--------------------------------------------------------
Kabhi kabhi SEQUENTIAL IDs zaroori hote hain, aur 
coordination cost acceptable hai. Ticket servers ek 
practical middle ground hain.

--------------------------------------------------------
KAISE KAAM KARTA
--------------------------------------------------------
Basic form mein, ticket server ek dedicated database hai 
jiska SIRF ek kaam hai: IDs distribute karna. Har request ek 
row insert karta aur naya auto-increment value return karta, 
isliye har caller ek DISTINCT, INCREASING number paata. 
Flickr ne MySQL ke saath ye approach use kiya:

```sql
-- Flickr-style ticket server
REPLACE INTO Tickets64 (stub) VALUES ('a');
SELECT LAST_INSERT_ID();
```

REPLACE auto-increment counter ADVANCE karta jabki table ek 
single row pe rehti (kabhi badhta nahi). Har call EXACTLY 
ek ID return karta.

NOTE: Backing store relational database hona zaroori nahi. 
Ek common in-memory alternative Redis hai (atomic INCR 
command), jo bina disk round-trip ke ek monotonically 
increasing counter return karta. Fast aur simple, but same 
trade-offs carry karta jo koi bhi central allocator karta: 
ek SINGLE POINT OF COORDINATION hai, aur persistence + 
replication carefully configure karni padti taaki restart 
counter values REPLAY ya LOSE na kare.

--------------------------------------------------------
BATCHING SE COORDINATION KAM KARNA
--------------------------------------------------------
Ek round-trip per ID ticket server ko write path pe HOTSPOT 
bana deta. Common optimization: EK BAAR MEIN RANGE handout 
karo. Server ek block claim karta (say 1,000 IDs), locally 
serve karta, aur ticket server se SIRF tab wapas jaata jab 
block exhaust ho jaaye.

TEXT DIAGRAM:

  App Server 1 -> Request ID block -> Ticket Server
  Ticket Server -> IDs 1-1000 -> App Server 1
  (App Server 1 uses IDs 1-1000 locally - koi coordination 
   nahi chahiye)
  
  App Server 2 -> Request ID block -> Ticket Server
  Ticket Server -> IDs 1001-2000 -> App Server 2
  (App Server 2 uses locally)
  
  App Server 1 -> Block exhausted, request more
  Ticket Server -> IDs 2001-3000 -> App Server 1

Batching coordination ko BLOCK SIZE ke factor se kam karta, 
but ek COST hai: agar server crash ho jaaye apne block mein 
IDs bache hue hon, wo IDs SKIP ho jaate.

--------------------------------------------------------
HIGH AVAILABILITY
--------------------------------------------------------
Single point of failure avoid karne ke liye, 2 ticket 
servers alag-alag OFFSETS ke saath chala sakte (jaise 
step-based auto-increment).

TRADE-OFFS:

  ACHA (PROS)                        BURA (CONS)
  -----------------------------------------------------
  Samajhna simple                    Central point of 
                                       coordination
  Sequential IDs                     Naye blocks ke liye 
                                       network latency
  Kisi bhi database ke saath kaam    Block allocation 
  karta                               complexity add karta
  Predictable capacity               Server crash mid-block 
                                       ho to IDs waste

Ye approach theek hai jab compact numeric IDs chahiye aur 
kuch coordination acceptable hai. Usually uniqueness aur 
rough monotonicity deta, but PERFECT gap-free global 
sequence nahi (kyunki unused IDs server crash mid-block se 
lost ho sakte).


NUMERIC IDs KO SHORT STRINGS MEIN ENCODE KARNA
--------------------------------------------------------
Auto-increment, Snowflake, aur ticket servers sab 64-bit 
integers produce karte. URL shortener ya kisi public-facing 
code ke liye, usually raw decimal number expose nahi karna 
chahte (jo LAMBA aur obviously sequential hota). BASE62 
ENCODING isko chhota kar deta.

Base62 ek number ko 62 characters se represent karta: 0-9, 
a-z, aur A-Z. Har character ~5.95 bits carry karta, isliye 
decimal se BAHUT ZYADA values per character encode karta.

EXAMPLE: 7-character Base62 code 62^7 (~3.5 trillion) 
values address kar sakta - zyada tar URL shorteners ke liye 
kaafi.

IMPORTANT CAVEAT: Base62 REVERSIBLE hai, isliye SEQUENTIAL 
counter encode karne se SEQUENTIAL, ENUMERABLE codes bante. 
Agar next code guess hone ki chinta hai, to underlying ID 
RANDOM scheme se generate karo, ya UNGUESSABLE public token 
apply karo - encoding pe order chhupane ke liye depend mat 
karo.


OTHER NOTABLE SCHEMES
--------------------------------------------------------

--------------------------------------------------------
KSUID (K-Sortable Unique Identifier)
--------------------------------------------------------
Segment ka format: 27 characters, 160 bits total.

KSUID custom epoch use karta aur Snowflake se BADA random 
component rakhta. Base62 encoded hai, isliye URL-safe hai.

--------------------------------------------------------
SONYFLAKE
--------------------------------------------------------
Sony ka Snowflake variation, LONGER LIFESPAN ke liye 
optimized:

Sonyflake THROUGHPUT ko LONGEVITY ke liye trade karta: 174 
saal ka IDs (69 ke bajaye), but sirf 256 IDs per 10ms 
(4,096 per ms ke bajaye). 65,536 machines support karta 
(1,024 ke bajaye).

--------------------------------------------------------
NANOID
--------------------------------------------------------
Short, URL-friendly IDs with customizable alphabet.

NanoID CRYPTOGRAPHICALLY SECURE hai, length aur character 
set configurable hai, aur public-facing random identifiers 
ke liye useful hai. Collision risk alphabet size aur chosen 
length pe depend karta.


SAHI APPROACH CHOOSE KARNA
--------------------------------------------------------
DECISION FRAMEWORK: Sabse important constraint se shuru 
karo, har sawal viable options ko narrow karta jab tak ek 
approach reh jaaye.

TEXT DIAGRAM - DECISION TREE:

  Need coordination-free generation?
    NO  -> Auto-Increment ya Ticket Server
    YES -> Need time-sortable IDs?
             NO  -> UUID v4
             YES -> Need 64-bit IDs specifically?
                      NO  -> ULID ya UUID v7
                      YES -> Can assign machine IDs?
                               YES -> Snowflake
                               NO  -> Ticket Server 
                                      (coordination accept 
                                      karo), ya 128-bit pe 
                                      relax karo: ULID/UUID v7

--------------------------------------------------------
DECISION FACTORS
--------------------------------------------------------
SCALE: Low scale ke liye, auto-increment usually theek. 
Medium scale ke liye, UUID/ULID/UUID v7/ticket servers achhe 
kaam karte. Very high scale + compact numeric IDs ke liye, 
Snowflake ya similar scheme consider karo. Exact thresholds 
database capacity, write distribution, aur ID generation 
critical path pe hai ya nahi, pe depend karte.

SORTABILITY: Feeds, time-series data, ya debugging ke liye 
time-ordered IDs chahiye, to Snowflake, ULID, ya UUID v7 use 
karo. Order matter nahi karta to UUID v4 simpler hai.

ID SIZE: Database compatibility, bandwidth, ya existing 
systems ke liye 64-bit IDs chahiye, to Snowflake ya 
auto-increment common options hain. 128 bits acceptable hai 
to UUID, UUID v7, ya ULID zyada flexibility dete.

INFRASTRUCTURE: Centralized database ke saath, 
auto-increment ya ticket servers easy hain. Distributed/
serverless environments mein, UUID ya ULID coordination 
avoid karte. Machine IDs reliably assign kar sakte ho, to 
Snowflake compact, time-ordered IDs ke liye strong fit hai.

SECURITY: IDs user-facing hain aur predictability concern 
hai, to UUID v4, NanoID, ya koi random-heavy scheme use 
karo. Sequential/time-ordered IDs creation time, volume 
expose kar sakte, ya enumeration allow kar sakte jab tak 
unguessable public token add na karo.

--------------------------------------------------------
COMMON PATTERNS IN PRACTICE
--------------------------------------------------------
URL SHORTENER: Counter-based ya Snowflake IDs with Base62 
encoding achhe kaam karte compact codes ke liye, but 
sequential public codes ENUMERABLE hote. Random slugs use 
karo ya collision handling add karo agar enumeration 
concern hai.

SOCIAL MEDIA POSTS: Snowflake-style IDs common hain kyunki 
time-sortability feeds aur storage locality mein help 
karti.

USER ACCOUNTS: Public IDs ke liye UUID v4 ya NanoID; 
internally Snowflake ya UUID v7 privacy considerations ke 
saath.

DATABASE PRIMARY KEYS: Single-primary systems ke liye 
auto-increment; UUID v7 ya ULID jab decentralized generation 
+ better locality useful ho. High-write workloads ke liye 
UUID v4 clustered primary key avoid karo jab tak database/
storage design random inserts absorb na kar sake.

API IDEMPOTENCY KEYS: UUID v4, typically client-generated.

DISTRIBUTED LOGS/EVENTS: ULID ya Snowflake time-sorted 
analysis ke liye.


COMPARISON SUMMARY TABLE
--------------------------------------------------------
  Approach       Size      Sortable  Coordination  Throughput  Best For
  --------------------------------------------------------------------------
  Auto-Increment 64 bits   Yes       Required      Medium      Single DB 
                                                                 systems
  UUID v4        128 bits  No        None          High        Distributed, 
                                                                 no ordering
  UUID v7        128 bits  Yes       None          High        Modern UUID 
                                                                 replacement
  Snowflake      64 bits   Yes       Machine ID    Very High   High-scale 
                                                                 services
  ULID           128 bits  Yes       None          High        General 
                                                                 purpose
  ObjectID       96 bits   Roughly   None          High        MongoDB
  Ticket Server  64 bits   Yes       Required      Medium      Sequential 
                                                                 requirement
  KSUID          160 bits  Yes       None          High        URL-safe, 
                                                                 sortable IDs
  Sonyflake      64 bits   Yes       Machine ID    Medium      Longer 
                                                                 lifespan than 
                                                                 Snowflake
  NanoID         Configurable No     None          High        Short, random 
                                                                 public IDs


OPERATIONAL CONSIDERATIONS
--------------------------------------------------------

--------------------------------------------------------
1. CLOCK SYNCHRONIZATION
--------------------------------------------------------
Time-based schemes clocks pe depend karte jo UNEXPECTEDLY 
backward nahi jaate. Time synchronization configured aur 
monitored hona chahiye. Decide karo clock backward jump 
kare to kya hoga: wait karo, fail closed karo, logical 
timestamp use karo, ya node ko service se hatao.

--------------------------------------------------------
2. MACHINE ID ASSIGNMENT
--------------------------------------------------------
Snowflake aur similar schemes ko UNIQUE machine IDs chahiye. 
Options:
- Configuration management (deployment ke dauran assign)
- ZooKeeper/etcd jaise coordination services (dynamic 
  assignment)
- IP address ya MAC address ke last N bits (sirf agar stable 
  aur collision-free ho tumhare environment mein)
- Cloud provider instance metadata

--------------------------------------------------------
3. COLLISION HANDLING
--------------------------------------------------------
Bahut kam collision probability ke saath bhi, DEFENSIVE 
measures add karo. Database mein UNIQUE CONSTRAINTS lagao, 
duplicate key errors gracefully handle karo, aur collisions 
pe log/alert karo - usually ye BUG, bad randomness, 
duplicated node IDs, ya clock issues indicate karta.

--------------------------------------------------------
4. MIGRATION
--------------------------------------------------------
Launch ke baad ID schemes badalna PAINFUL hai. Shuru mein 
carefully choose karo. Agar migrate karna zaroori ho:
- Purane aur naye IDs ke beech ek MAPPING TABLE maintain 
  karo
- Kuch schemes different prefixes ke saath COEXIST kar sakte
- External references/URLs ke liye purane IDs resolvable 
  rakho
- Ek LAMBI TRANSITION PERIOD ka plan banao


SUMMARY - KEY TAKEAWAYS
--------------------------------------------------------
Koi "universally best" ID scheme nahi hai. Sahi choice 
requirements se nikalti hai jo tum shuru mein clarify karte 
ho: uniqueness scope, ordering needs, ID length, 
predictability, aur write path pe coordination acceptable 
hai kya. Comparison Summary table har scheme ko in 
properties se map karti; ise use karo apni choice ko 
JUSTIFY karne ke liye, sirf ek default naam lene ke bajaye.

- Modern distributed systems ke liye, ULID ya UUID v7 
  usually achha balance dete uniqueness, sortability, aur 
  simplicity ka, bina per-ID coordination ke.
- High-scale systems jaha 64-bit IDs zaroori hain (database 
  performance, network bandwidth, ya compatibility ke liye), 
  Snowflake ya similar timestamp-node-sequence scheme 
  usually best fit hoti hai.
- Jo bhi choose karo, JALDI karo. Running production system 
  mein ID schemes badalna painful hai kyunki IDs URLs, logs, 
  APIs, analytics, aur foreign keys mein LEAK ho jaate hain.

========================================================
END OF NOTES
========================================================
*/